In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
import os
from io import BytesIO

import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv()

BUCKET_NAME = os.environ["MINIO_BUCKET"]

s3_client = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
    region_name="us-east-1",
    config=Config(signature_version="s3v4"),
)

In [5]:
def read_parquet_prefix(s3_client, bucket_name, prefix):
    """Read every Parquet object under a MinIO prefix."""
    paginator = s3_client.get_paginator("list_objects_v2")
    parquet_keys = []

    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith(".parquet"):
                parquet_keys.append(key)

    if not parquet_keys:
        raise FileNotFoundError(
            f"No Parquet files found under prefix: {prefix}"
        )

    dataframes = []
    for key in parquet_keys:
        response = s3_client.get_object(Bucket=bucket_name, Key=key)
        dataframe = pd.read_parquet(
            BytesIO(response["Body"].read()),
            engine="fastparquet",
        )
        dataframes.append(dataframe)
        print(f"Loaded: {key}")
        print(f"Shape: {dataframe.shape}")

    return pd.concat(dataframes, ignore_index=True)


def read_parquet_object(s3_client, bucket_name, object_key):
    response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
    return pd.read_parquet(
        BytesIO(response["Body"].read()),
        engine="fastparquet",
    )

In [6]:
metadata_df = read_parquet_prefix(
    s3_client,
    BUCKET_NAME,
    "metadata/etl_metadata/"
)

required_silver_datasets = {"train_data", "test_data", "RUL_data"}
successful_silver_metadata = metadata_df[
    (metadata_df["layer"] == "silver")
    & (metadata_df["status"] == "success")
    & metadata_df["dataset_name"].isin(required_silver_datasets)
]

datasets_by_run = successful_silver_metadata.groupby("run_id")["dataset_name"].apply(set)
valid_silver_run_ids = sorted(
    run_id
    for run_id, datasets in datasets_by_run.items()
    if required_silver_datasets.issubset(datasets)
)

if not valid_silver_run_ids:
    raise FileNotFoundError(
        "No successful Silver run contains train, test, and RUL data."
    )

pipeline_run_id = valid_silver_run_ids[-1]
selected_silver_metadata = successful_silver_metadata[
    successful_silver_metadata["run_id"] == pipeline_run_id
]
pipeline_load_date = selected_silver_metadata["load_date"].iloc[0]
metadata_key = (
    f"metadata/etl_metadata/load_date={pipeline_load_date}/"
    f"run_id={pipeline_run_id}/manifest.parquet"
)

def silver_object_key(dataset_name):
    matches = selected_silver_metadata[
        selected_silver_metadata["dataset_name"] == dataset_name
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one Silver object for {dataset_name}.")
    return matches.iloc[0]["object_key"]

original_train_df_minio = read_parquet_object(
    s3_client, BUCKET_NAME, silver_object_key("train_data")
)
original_test_df_minio = read_parquet_object(
    s3_client, BUCKET_NAME, silver_object_key("test_data")
)
original_rul_df_minio = read_parquet_object(
    s3_client, BUCKET_NAME, silver_object_key("RUL_data")
)

print(f"Selected Silver run: {pipeline_run_id}")

Loaded: metadata/etl_metadata/load_date=2026-09-02/run_id=20260902T120041719003Z/manifest.parquet
Shape: (7, 8)
Selected Silver run: 20260902T120041719003Z


In [7]:
pd.set_option('display.max_columns', None)


def prepare_feature_input(dataframe):
    dataframe = dataframe.copy()

    dataframe = dataframe.drop(
        columns=["processing_timestamp", "data_quality_status"]
    )

    for column in dataframe.columns:
        if column not in [
            "unit_number",
            "time_in_cycles",
            "operational_setting_1",
            "operational_setting_2",
            "operational_setting_3",
        ]:
            dataframe = dataframe.rename(
                columns={column: "current_" + column}
            )

    return dataframe.sort_values(
        ["unit_number", "time_in_cycles"]
    ).reset_index(drop=True)


train_df_minio = prepare_feature_input(original_train_df_minio)
test_df_minio = prepare_feature_input(original_test_df_minio)
rul_df_minio = original_rul_df_minio.copy().drop(
    columns=["processing_timestamp", "data_quality_status"]
)

sensors = [
    column
    for column in train_df_minio.columns
    if column not in [
        "unit_number",
        "time_in_cycles",
        "operational_setting_1",
        "operational_setting_2",
        "operational_setting_3",
    ]
]

if sensors != [
    column
    for column in test_df_minio.columns
    if column not in [
        "unit_number",
        "time_in_cycles",
        "operational_setting_1",
        "operational_setting_2",
        "operational_setting_3",
    ]
]:
    raise ValueError("Train and test sensor columns do not match.")

In [8]:
def add_time_series_features(dataframe, sensors):
    dataframe = dataframe.copy()

    rolling_means = (
        dataframe.groupby(["unit_number"])[sensors]
        .rolling(window=20, min_periods=20)
        .mean()
        .reset_index(level=0, drop=True)
    )
    rolling_means = rolling_means.rename(
        columns={
            column: column[8:] + "_mean_20"
            for column in rolling_means.columns
        }
    )
    dataframe = pd.concat([dataframe, rolling_means], axis=1)

    rolling_stds = (
        dataframe.groupby("unit_number")[sensors]
        .rolling(window=20, min_periods=20)
        .std()
        .reset_index(level=0, drop=True)
    )
    rolling_stds = rolling_stds.rename(
        columns={
            column: column[8:] + "_std_20"
            for column in rolling_stds.columns
        }
    )
    dataframe = pd.concat([dataframe, rolling_stds], axis=1)

    rolling_min = (
        dataframe.groupby("unit_number")[sensors]
        .rolling(window=20, min_periods=20)
        .min()
        .reset_index(level=0, drop=True)
    )
    rolling_min = rolling_min.rename(
        columns={
            column: column[8:] + "_min_20"
            for column in rolling_min.columns
        }
    )
    dataframe = pd.concat([dataframe, rolling_min], axis=1)

    rolling_max = (
        dataframe.groupby("unit_number")[sensors]
        .rolling(window=20, min_periods=20)
        .max()
        .reset_index(level=0, drop=True)
    )
    rolling_max = rolling_max.rename(
        columns={
            column: column[8:] + "_max_20"
            for column in rolling_max.columns
        }
    )
    dataframe = pd.concat([dataframe, rolling_max], axis=1)

    difference = dataframe.groupby("unit_number")[sensors].diff()
    difference = difference.rename(
        columns={
            column: column[8:] + "_difference"
            for column in difference.columns
        }
    )
    dataframe = pd.concat([dataframe, difference], axis=1)

    percentage_change = (
        dataframe.groupby("unit_number")[sensors]
        .pct_change(fill_method=None)
        .replace([np.inf, -np.inf], np.nan)
    )
    percentage_change = percentage_change.rename(
        columns={
            column: column[8:] + "_%change"
            for column in percentage_change.columns
        }
    )
    dataframe = pd.concat([dataframe, percentage_change], axis=1)

    sensor_slope = (
        dataframe.groupby("unit_number")[sensors]
        .rolling(window=20, min_periods=20)
        .apply(
            lambda x: np.polyfit(np.arange(len(x)), x, 1)[0],
            raw=True,
        )
        .reset_index(level=0, drop=True)
    )
    sensor_slope = sensor_slope.rename(
        columns={
            column: column[8:] + "_sensor_slope_20"
            for column in sensor_slope.columns
        }
    )
    dataframe = pd.concat([dataframe, sensor_slope], axis=1)

    ema_features = (
        dataframe.groupby("unit_number")[sensors]
        .transform(
            lambda column: column.ewm(
                span=20,
                adjust=False,
                min_periods=1,
            ).mean()
        )
    )
    ema_features = ema_features.rename(
        columns={
            column: column[8:] + "_ema_20"
            for column in ema_features.columns
        }
    )
    dataframe = pd.concat([dataframe, ema_features], axis=1)

    return dataframe

In [9]:
train_df_minio = add_time_series_features(train_df_minio, sensors)
test_df_minio = add_time_series_features(test_df_minio, sensors)

print(f"Train shape after feature engineering: {train_df_minio.shape}")
print(f"Test shape after feature engineering: {test_df_minio.shape}")

Train shape after feature engineering: (20631, 194)
Test shape after feature engineering: (13096, 194)


In [10]:
temp1 =  train_df_minio.groupby("unit_number").agg(last_cycle=("time_in_cycles","max"))
train_df_minio = train_df_minio.merge(temp1,on="unit_number",how="left")
train_df_minio["remaining_useful_life"] = train_df_minio["last_cycle"]-train_df_minio["time_in_cycles"]
train_df_minio = train_df_minio.drop(columns=["last_cycle"])

In [11]:
train_df_minio.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,current_sensor_measurement_1,current_sensor_measurement_2,current_sensor_measurement_3,current_sensor_measurement_4,current_sensor_measurement_5,current_sensor_measurement_6,current_sensor_measurement_7,current_sensor_measurement_8,current_sensor_measurement_9,current_sensor_measurement_10,current_sensor_measurement_11,current_sensor_measurement_12,current_sensor_measurement_13,current_sensor_measurement_14,current_sensor_measurement_15,current_sensor_measurement_16,current_sensor_measurement_17,current_sensor_measurement_18,current_sensor_measurement_19,current_sensor_measurement_20,current_sensor_measurement_21,sensor_measurement_1_mean_20,sensor_measurement_2_mean_20,sensor_measurement_3_mean_20,sensor_measurement_4_mean_20,sensor_measurement_5_mean_20,sensor_measurement_6_mean_20,sensor_measurement_7_mean_20,sensor_measurement_8_mean_20,sensor_measurement_9_mean_20,sensor_measurement_10_mean_20,sensor_measurement_11_mean_20,sensor_measurement_12_mean_20,sensor_measurement_13_mean_20,sensor_measurement_14_mean_20,sensor_measurement_15_mean_20,sensor_measurement_16_mean_20,sensor_measurement_17_mean_20,sensor_measurement_18_mean_20,sensor_measurement_19_mean_20,sensor_measurement_20_mean_20,sensor_measurement_21_mean_20,sensor_measurement_1_std_20,sensor_measurement_2_std_20,sensor_measurement_3_std_20,sensor_measurement_4_std_20,sensor_measurement_5_std_20,sensor_measurement_6_std_20,sensor_measurement_7_std_20,sensor_measurement_8_std_20,sensor_measurement_9_std_20,sensor_measurement_10_std_20,sensor_measurement_11_std_20,sensor_measurement_12_std_20,sensor_measurement_13_std_20,sensor_measurement_14_std_20,sensor_measurement_15_std_20,sensor_measurement_16_std_20,sensor_measurement_17_std_20,sensor_measurement_18_std_20,sensor_measurement_19_std_20,sensor_measurement_20_std_20,sensor_measurement_21_std_20,sensor_measurement_1_min_20,sensor_measurement_2_min_20,sensor_measurement_3_min_20,sensor_measurement_4_min_20,sensor_measurement_5_min_20,sensor_measurement_6_min_20,sensor_measurement_7_min_20,sensor_measurement_8_min_20,sensor_measurement_9_min_20,sensor_measurement_10_min_20,sensor_measurement_11_min_20,sensor_measurement_12_min_20,sensor_measurement_13_min_20,sensor_measurement_14_min_20,sensor_measurement_15_min_20,sensor_measurement_16_min_20,sensor_measurement_17_min_20,sensor_measurement_18_min_20,sensor_measurement_19_min_20,sensor_measurement_20_min_20,sensor_measurement_21_min_20,sensor_measurement_1_max_20,sensor_measurement_2_max_20,sensor_measurement_3_max_20,sensor_measurement_4_max_20,sensor_measurement_5_max_20,sensor_measurement_6_max_20,sensor_measurement_7_max_20,sensor_measurement_8_max_20,sensor_measurement_9_max_20,sensor_measurement_10_max_20,sensor_measurement_11_max_20,sensor_measurement_12_max_20,sensor_measurement_13_max_20,sensor_measurement_14_max_20,sensor_measurement_15_max_20,sensor_measurement_16_max_20,sensor_measurement_17_max_20,sensor_measurement_18_max_20,sensor_measurement_19_max_20,sensor_measurement_20_max_20,sensor_measurement_21_max_20,sensor_measurement_1_difference,sensor_measurement_2_difference,sensor_measurement_3_difference,sensor_measurement_4_difference,sensor_measurement_5_difference,sensor_measurement_6_difference,sensor_measurement_7_difference,sensor_measurement_8_difference,sensor_measurement_9_difference,sensor_measurement_10_difference,sensor_measurement_11_difference,sensor_measurement_12_difference,sensor_measurement_13_difference,sensor_measurement_14_difference,sensor_measurement_15_difference,sensor_measurement_16_difference,sensor_measurement_17_difference,sensor_measurement_18_difference,sensor_measurement_19_difference,sensor_measurement_20_difference,sensor_measurement_21_difference,sensor_measurement_1_%change,sensor_measurement_2_%change,sensor_measurement_3_%change,sensor_measurement_4_%change,sensor_measurement_5_%change,sensor_measurement_6_%change,sensor

In [12]:
from datetime import datetime, timezone
gold_dataframes = {
    "train_features": train_df_minio,
    "test_features": test_df_minio,
}
gold_metadata_records = []

for dataset_name, dataframe in gold_dataframes.items():
    parquet_buffer = BytesIO()

    dataframe.to_parquet(
        parquet_buffer,
        index=False,
        engine="fastparquet",
    )

    parquet_buffer.seek(0)
    object_key = (
        f"gold/{dataset_name}/"
        f"load_date={pipeline_load_date}/"
        f"run_id={pipeline_run_id}/"
        f"{dataset_name}.parquet"
    )

    s3_client.put_object(
        Bucket=BUCKET_NAME,
        Key=object_key,
        Body=parquet_buffer.getvalue(),
        ContentType="application/octet-stream",
    )

    print(f"Uploaded Gold dataset to MinIO: {object_key}")
    gold_metadata_records.append({
        "run_id": pipeline_run_id,
        "load_date": pipeline_load_date,
        "layer": "gold",
        "dataset_name": dataset_name,
        "object_key": object_key,
        "status": "success",
        "row_count": len(dataframe),
        "recorded_at": datetime.now(timezone.utc).isoformat(),
    })

etl_metadata_df = read_parquet_object(
    s3_client, BUCKET_NAME, metadata_key
)
etl_metadata_df = etl_metadata_df[
    ~(
        (etl_metadata_df["run_id"] == pipeline_run_id)
        & (etl_metadata_df["layer"] == "gold")
        & etl_metadata_df["dataset_name"].isin(
            {"train_features", "test_features"}
        )
    )
]
etl_metadata_df = pd.concat(
    [etl_metadata_df, pd.DataFrame(gold_metadata_records)],
    ignore_index=True,
)
metadata_buffer = BytesIO()
etl_metadata_df.to_parquet(
    metadata_buffer,
    index=False,
    engine="fastparquet",
)
metadata_buffer.seek(0)
s3_client.put_object(
    Bucket=BUCKET_NAME,
    Key=metadata_key,
    Body=metadata_buffer.getvalue(),
    ContentType="application/octet-stream",
)
print(f"Updated ETL metadata manifest: {metadata_key}")

Uploaded Gold dataset to MinIO: gold/train_features/load_date=2026-09-02/run_id=20260902T120041719003Z/train_features.parquet
Uploaded Gold dataset to MinIO: gold/test_features/load_date=2026-09-02/run_id=20260902T120041719003Z/test_features.parquet
Updated ETL metadata manifest: metadata/etl_metadata/load_date=2026-09-02/run_id=20260902T120041719003Z/manifest.parquet
